# Graph Fraud Detection — OFS Setup

Initializes the Feature Store layer on top of the existing graph-fraud infrastructure.

| Step | What happens |
|------|------|
| 1 | Connect & configure session |
| 2 | Create producer/consumer roles |
| 3 | Initialize Feature Store + Online Service (Postgres) |
| 4 | Register 3 entities: APPLICANT, ADDRESS, IP_NETWORK |
| 5 | Resume Leiden refresh task |

**Prerequisites**
- `setup/01_foundations.sql` — raw data (APPLICATIONS, APPLICANTS, etc.)
- `setup/02_edge_construction.sql` — EDGES.APPLICANT_EDGES dynamic table
- `setup/03_spcs_infrastructure.sql` — GPU pool, COMMUNITY_RESULTS, Leiden job
- `setup/04_community_assignments.sql` — COMMUNITY_ASSIGNMENTS table
- `SNOWFLAKE_PAT` env var or Snowsight active session
- `snowflake-ml-python >= 1.41`

In [1]:
import os
import time

import snowflake.connector
from snowflake.snowpark import Session
from snowflake.ml.feature_store import (
    FeatureStore,
    Entity,
    CreationMode,
)

from importlib.metadata import version
print(f'snowflake-ml-python: {version("snowflake-ml-python")}')

# Load PAT from Snowflake config
import tomllib, pathlib
_sf_config = tomllib.load(open(pathlib.Path.home() / '.snowflake' / 'config.toml', 'rb'))
os.environ.setdefault('SNOWFLAKE_PAT', _sf_config['connections']['parker_demo']['password'])
print(f'SNOWFLAKE_PAT loaded ({len(os.environ["SNOWFLAKE_PAT"])} chars)')


snowflake-ml-python: 2.4.0
SNOWFLAKE_PAT loaded (218 chars)


## 1. Configuration & Connection

In [2]:
CONNECTION_NAME   = 'parker_demo'
DATABASE          = 'GRAPH_FRAUD_DEMO'
FEATURE_SCHEMA    = 'FEATURES'
WAREHOUSE         = 'GRAPH_FRAUD_WH'
PRODUCER_ROLE     = 'GRAPH_FRAUD_PRODUCER'
CONSUMER_ROLE     = 'GRAPH_FRAUD_CONSUMER'
PARENT_ROLE       = 'ACCOUNTADMIN'

In [3]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    conn = snowflake.connector.connect(connection_name=CONNECTION_NAME)
    session = Session.builder.configs({'connection': conn}).create()

session.sql_simplifier_enabled = False
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()
print(f'Account: {session.get_current_account()}')

You might have more than one threads sharing the Session object trying to update sql_simplifier_enabled. Updating this while other tasks are running can potentially cause unexpected behavior. Please update the session configuration before starting the threads.


Account: "SFSENORTHAMERICA-PERICKSON_AWS1"


## 2. Producer & Consumer Roles

The Online Service needs a producer role (writes features) and a consumer role (reads features).
Both are granted to ACCOUNTADMIN for this demo.

In [4]:
for role in [PRODUCER_ROLE, CONSUMER_ROLE]:
    session.sql(f'CREATE ROLE IF NOT EXISTS {role}').collect()
    session.sql(f'GRANT ROLE {role} TO ROLE {PARENT_ROLE}').collect()
    print(f'  Created and granted: {role}')

  Created and granted: GRAPH_FRAUD_PRODUCER
  Created and granted: GRAPH_FRAUD_CONSUMER


## 3. Initialize Feature Store & Online Service

The Online Service is a managed Postgres-backed serving layer (<20 ms p50).
Created once per Feature Store — provisions two REST endpoints:
- **Ingest** — accepts streaming filing events
- **Query** — ultra-low-latency feature serving

First creation takes several minutes; we poll until `RUNNING`.

In [5]:
fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=FEATURE_SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print(f'Feature Store initialized: {DATABASE}.{FEATURE_SCHEMA}')

Feature Store initialized: GRAPH_FRAUD_DEMO.FEATURES


In [6]:
from snowflake.ml.feature_store import online_service

try:
    fs.create_online_service(PRODUCER_ROLE, CONSUMER_ROLE)
    print('Online Service creation initiated...')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Online Service already exists \u2014 skipping creation.')
    else:
        raise

query_url = ingest_url = None
for i in range(40):
    status = fs.get_online_service_status()
    ep_names = [ep.name for ep in status.endpoints] if status.endpoints else []
    print(f'  [{i:02d}] Status: {status.status} | Endpoints: {ep_names}')
    if status.status == 'RUNNING' and online_service.endpoint_url(status, 'query'):
        query_url  = online_service.endpoint_url(status, 'query')
        ingest_url = online_service.endpoint_url(status, 'ingest')
        print(f'\nOnline Service RUNNING')
        print(f'  Query  : {query_url}')
        print(f'  Ingest : {ingest_url}')
        break
    time.sleep(30)
else:
    print('WARNING: Online Service not RUNNING after 20 min. Check Snowsight.')

Online Service creation initiated...
  [00] Status: STARTING | Endpoints: ['ingest', 'query']
  [01] Status: STARTING | Endpoints: ['ingest', 'query']
  [02] Status: STARTING | Endpoints: ['ingest', 'query']
  [03] Status: STARTING | Endpoints: ['ingest', 'query']
  [04] Status: STARTING | Endpoints: ['ingest', 'query']
  [05] Status: STARTING | Endpoints: ['ingest', 'query']
  [06] Status: RUNNING | Endpoints: ['ingest', 'query']

Online Service RUNNING
  Query  : https://bvciccnd-sfsenorthamerica-perickson-aws1.snowflakecomputing.app
  Ingest : https://nuciccnd-sfsenorthamerica-perickson-aws1.snowflakecomputing.app


## 4. Register Entities

Three entities capture the graph-fraud domain:

| Entity | Join Key | Purpose |
|---|---|---|
| APPLICANT | APPLICANT_ID | Individual applicant — direct graph features |
| ADDRESS | ADDRESS_ID | Filing address — inherited community context |
| IP_NETWORK | IP_ADDRESS | IP address — inherited community context |

In [7]:
applicant_entity = Entity(
    name='APPLICANT',
    join_keys=['APPLICANT_ID'],
    desc='Individual applicant',
)
fs.register_entity(applicant_entity)
print('Registered: APPLICANT')

address_entity = Entity(
    name='ADDRESS',
    join_keys=['ADDRESS_ID'],
    desc='Filing address \u2014 inherited community context for new applicants',
)
fs.register_entity(address_entity)
print('Registered: ADDRESS')

ip_entity = Entity(
    name='IP_NETWORK',
    join_keys=['IP_ADDRESS'],
    desc='IP address \u2014 inherited community context for new applicants',
)
fs.register_entity(ip_entity)
print('Registered: IP_NETWORK')

fs.list_entities().show()

Registered: APPLICANT
Registered: ADDRESS
Registered: IP_NETWORK
----------------------------------------------------------------------------------------------------------
|"NAME"      |"JOIN_KEYS"            |"DESC"                                              |"OWNER"       |
----------------------------------------------------------------------------------------------------------
|ADDRESS     |["ADDRESS_ID"]  |Filing address — inherited community context fo...  |ACCOUNTADMIN  |
|IP_NETWORK  |["IP_ADDRESS"]         |IP address — inherited community context for ne...  |ACCOUNTADMIN  |
|APPLICANT    |["APPLICANT_ID"]        |Individual applicant                           |ACCOUNTADMIN  |
----------------------------------------------------------------------------------------------------------



## 5. Resume Leiden Refresh Task

The Leiden job is scheduled every 15 minutes (created suspended in `03_spcs_infrastructure.sql`).
Resume it so community assignments stay fresh.

In [8]:
session.sql('ALTER TASK IF EXISTS GRAPH.LEIDEN_REFRESH_TASK RESUME').collect()
print('Leiden refresh task resumed (15-min schedule)')

Leiden refresh task resumed (15-min schedule)


## 6. Verify Setup

In [9]:
from snowflake.ml.feature_store import online_service

print('=== Setup Verification ===')
print(f'  Database       : {DATABASE}')
print(f'  Feature Schema : {FEATURE_SCHEMA}')
print(f'  Warehouse      : {WAREHOUSE}')

for tbl in ['RAW.APPLICATIONS', 'RAW.APPLICANTS', 'GRAPH.COMMUNITY_RESULTS']:
    cnt = session.table(f'{DATABASE}.{tbl}').count()
    print(f'  {tbl:30s}: {cnt:>12,} rows')

status = fs.get_online_service_status()
print(f'  Online Service : {status.status}')
if online_service.endpoint_url(status, 'query'):
    print(f'    Query  : {online_service.endpoint_url(status, "query")}')
    print(f'    Ingest : {online_service.endpoint_url(status, "ingest")}')

print('\nProceed to 01_features.ipynb')

=== Setup Verification ===
  Database       : GRAPH_FRAUD_DEMO
  Feature Schema : FEATURES
  Warehouse      : GRAPH_FRAUD_WH
  RAW.APPLICATIONS               :    3,000,001 rows
  RAW.APPLICANTS                 :    3,000,000 rows
  GRAPH.COMMUNITY_RESULTS       :    2,980,028 rows
  Online Service : RUNNING
    Query  : https://bvciccnd-sfsenorthamerica-perickson-aws1.snowflakecomputing.app
    Ingest : https://nuciccnd-sfsenorthamerica-perickson-aws1.snowflakecomputing.app

Proceed to 01_features.ipynb
